# Alpaca collection with additional checks

This version builds on Carmen's collection notebook and the group's earlier API exploration. It adds resumable downloads, checks on saved files and a price-table export. Carmen's original Alpaca cells are preserved in `collect_data.ipynb`, which also has an added Yahoo section.

Collection is off by default. With `RUN_COLLECTION = False`, the notebook shows the plan without loading API keys or making requests. To collect, set it to `True` and use this project's `.env` file.

The defaults request AAPL and JNJ headlines and AAPL, JNJ and SPY daily prices, using SIP with all corporate-action adjustments. If that feed is unavailable to the account, the request stops so the error can be checked.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Works when the notebook starts in either the project folder or src/.
PROJECT_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "src" / "collect_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the finmedia-stock-pred project folder.")

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from collect_pipeline import CollectionConfig, collect_dataset, plan_collection


## Requested data

The start time is included and the end time is excluded. These bounds use UTC, so the end does not include the full New York calendar day of October 16.

The Alpaca SDK handles pagination. The collector leaves the total-result limit unset and divides news into non-overlapping yearly intervals.

In [ ]:
RUN_COLLECTION = False

CONFIG = CollectionConfig(
    tickers=("AAPL", "JNJ"),
    benchmark="SPY",
    start="2020-01-01T00:00:00Z",
    end="2026-10-16T00:00:00Z",
    price_feed="sip",
    price_adjustment="all",
    max_attempts=3,
)

PLAN = plan_collection(PROJECT_ROOT, CONFIG)
PLAN


## Running collection

When collection is enabled, API keys are read from this project's `.env` file without printing them. A new request is made only if a saved chunk is missing or fails validation.

Each chunk has a manifest recording its request settings, package versions, columns, row count, time range and file hash. Completed chunks, including empty responses, can be reused after an interrupted run. Temporary failures have a limited number of retries.

In [3]:
collection_result = None

if RUN_COLLECTION:
    def show_progress(event):
        label = event.get("ticker", "prices")
        print(f"{label}: {event['action']}, {event['rows']:,} rows "
              f"({event['cache_reason']})")

    collection_result = collect_dataset(
        PROJECT_ROOT,
        CONFIG,
        progress=show_progress,
    )
    display(pd.DataFrame(collection_result["events"]))
else:
    print("Collection is off. No API credentials were loaded and no new API requests were made.")


Collection is off. No API credentials were loaded and no new API requests were made.


## Saved files

- `data/raw/news/AAPL.parquet` and `JNJ.parquet`: combined news with article IDs, URLs and publication, update and collection times.
- `data/raw/news/chunks/`: returned records stored by year.
- `data/raw/prices.parquet`: daily prices, including SPY.
- Matching `.manifest.json` files and `data/raw/collection_run.json`: settings, file checks and run status.

The combined files keep the latest returned version of each ticker/article-ID pair. Records without IDs are retained. An article can appear under both stocks. Update times help identify revisions, but do not reconstruct exactly what a headline said at an earlier prediction time.

Completed downloads still need coverage checks. An empty response alone does not tell us whether news was absent or missing from the provider's archive.

In [4]:
if collection_result is not None:
    collection_summary = [
        {"dataset": ticker, "rows": values["rows"], "path": values["path"]}
        for ticker, values in collection_result["news"].items()
    ]
    collection_summary.append({
        "dataset": "prices (stocks + benchmark)",
        "rows": collection_result["prices"]["rows"],
        "path": collection_result["prices"]["path"],
    })
    display(pd.DataFrame(collection_summary))


## Yahoo daily prices

This section can run on its own. It requests AAPL, AMZN, JNJ and SPY prices through Yahoo's public chart interface, which is undocumented and may change. Files are saved separately from Alpaca.

The range uses New York trading dates: `2020-01-01 <= date < 2026-10-16`. The last included session is October 15. Request boundaries are converted from New York midnight to Unix seconds.

With `RUN_YAHOO_COLLECTION = False`, only the local cache is checked. Set it to `True` to allow requests for missing or invalid files. This section does not use Alpaca keys.

In [ ]:
from pathlib import Path
from datetime import date, datetime, time as date_time
from zoneinfo import ZoneInfo
import sys
import pandas as pd
import requests
from IPython.display import display

YAHOO_ROOT = next(
    folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "src" / "collect_public_prices.py").is_file()
)
if str(YAHOO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(YAHOO_ROOT / "src"))

from collect_public_prices import (
    PublicPriceConfig, run_collection,
    ENDPOINT as YAHOO_HELPER_ENDPOINT,
)

RUN_YAHOO_COLLECTION = False
YAHOO_CONFIG = PublicPriceConfig(
    symbols=("AAPL", "JNJ", "SPY", "AMZN"),
    start="2020-01-01",
    end="2026-10-16",  # Exclusive New York session-date boundary.
)
YAHOO_ENDPOINT = "https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
YAHOO_TIMEZONE = ZoneInfo("America/New_York")
YAHOO_PARAMS = {
    "period1": int(datetime.combine(
        date.fromisoformat(YAHOO_CONFIG.start), date_time.min,
        tzinfo=YAHOO_TIMEZONE,
    ).timestamp()),
    "period2": int(datetime.combine(
        date.fromisoformat(YAHOO_CONFIG.end), date_time.min,
        tzinfo=YAHOO_TIMEZONE,
    ).timestamp()),
    "interval": "1d",
    "events": "div,splits",
    "includeAdjustedClose": "true",
}
assert YAHOO_ENDPOINT == YAHOO_HELPER_ENDPOINT
assert YAHOO_PARAMS == YAHOO_CONFIG.parameters()

display(pd.DataFrame([
    {"symbol": symbol, "endpoint": YAHOO_ENDPOINT.format(symbol=symbol), **YAHOO_PARAMS}
    for symbol in YAHOO_CONFIG.symbols
]))


In [ ]:
# This is the HTTP call used by the collector when a request is needed.
# The helper handles validation, bounded retries, raw-response saving, and caches.
class YahooNotebookHTTP:
    def get(self, url, *, params, headers, timeout, allow_redirects):
        return requests.get(
            url,
            params=params,
            headers=headers,
            timeout=timeout,
            allow_redirects=allow_redirects,
        )


if RUN_YAHOO_COLLECTION:
    yahoo_collection_result = run_collection(
        YAHOO_ROOT,
        YAHOO_CONFIG,
        collect=True,
        session=YahooNotebookHTTP(),
    )
    print("Collection status:", yahoo_collection_result["status"])
else:
    print("Yahoo collection is off: no HTTP requests are made.")

# Local inspection checks request parameters, versions, and stored file hashes.
yahoo_cache_status = run_collection(YAHOO_ROOT, YAHOO_CONFIG, collect=False)
print("Validated-cache status:", yahoo_cache_status["status"])
display(pd.DataFrame(yahoo_cache_status["symbols"]).T)


In [ ]:
YAHOO_PRICE_FILE = YAHOO_ROOT / "data/supplemental/yahoo_prices/prices.parquet"

if yahoo_cache_status["status"] == "ready":
    yahoo_local_prices = pd.read_parquet(YAHOO_PRICE_FILE)
    print("Local Yahoo price rows:", f"{len(yahoo_local_prices):,}")
    display(yahoo_local_prices.groupby("symbol").agg(
        rows=("date", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    ))
    display(yahoo_local_prices.head(8))
    display(pd.DataFrame({
        "column": yahoo_local_prices.columns,
        "dtype": yahoo_local_prices.dtypes.astype(str).to_numpy(),
        "missing_values": yahoo_local_prices.isna().sum().to_numpy(),
    }))
else:
    print("No fully validated Yahoo dataset is available yet; enable collection above if needed.")


Yahoo responses, price tables and manifests are saved under `data/supplemental/yahoo_prices/`. The supplied OHLC and volume values are kept alongside a separate `adjusted_close` column. No extra price adjustment is applied here.

The separate news collector is `collect_public_news.py`.